<a href="https://colab.research.google.com/github/lgsilva-dev/data-science-fundamentals/blob/main/extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd
import time
import sys

# ==========================================
# 1. CONFIGURAÇÕES DE AUTENTICAÇÃO
# ==========================================
CLIENT_ID = "f2093dbfbf554971805439720e1b4d29"
CLIENT_SECRET = "471b12a2a7f54e3d811b885ba7d2465f"

# Atualizado para o loopback IPv4 exigido pelo Spotify
REDIRECT_URI = "http://127.0.0.1:8888/callback"

# ADICIONADO: user-library-read para ler as "Músicas Curtidas"
SCOPE = "playlist-read-private playlist-read-collaborative user-library-read user-read-private"

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=REDIRECT_URI,
                                               scope=SCOPE,
                                               open_browser=False))

# ==========================================
# 2. FUNÇÕES DE EXTRAÇÃO
# ==========================================
def get_user_playlists():
    """Busca APENAS as playlists criadas pelo próprio usuário."""
    # Primeiro, descobrimos o seu ID interno no Spotify
    me = sp.current_user()
    my_user_id = me['id']
    print(f"\nAutenticado como: {me.get('display_name', my_user_id)}")

    playlist_data = []
    limit = 50
    offset = 0

    while True:
        playlists = sp.current_user_playlists(limit=limit, offset=offset)
        if not playlists['items']:
            break

        for playlist in playlists['items']:
            if not playlist:
                continue

            # FILTRO DE DADOS: Só adiciona se o dono da playlist for você
            if playlist['owner']['id'] == my_user_id:
                playlist_data.append({
                    'playlist_id': playlist['id'],
                    'playlist_name': playlist['name']
                })
        offset += limit

    print(f"Total de playlists PRÓPRIAS encontradas: {len(playlist_data)}")
    return playlist_data

def get_tracks_from_playlists(playlists):
    """Extrai músicas das suas playlists."""
    tracks_data = []

    for pl in playlists:
        limit = 100
        offset = 0

        while True:
            try:
                results = sp.playlist_tracks(pl['playlist_id'], limit=limit, offset=offset)
            except spotipy.exceptions.SpotifyException as e:
                # O erro detalhado ajudará a entender se a API ainda está reclamando de algo
                print(f"⚠️ Pulando '{pl['playlist_name']}' (Erro da API: {e.http_status})")
                break

            tracks = results['items']
            if not tracks:
                break

            for item in tracks:
                track = item.get('track')
                if not track or track.get('is_local') or not track.get('id'):
                    continue

                artist_id = track['artists'][0]['id'] if track.get('artists') else None
                tracks_data.append({
                    'source': f"Playlist: {pl['playlist_name']}",
                    'track_id': track['id'],
                    'track_name': track['name'],
                    'artist_id': artist_id,
                    'artist_name': track['artists'][0]['name'] if track.get('artists') else "Unknown"
                })
            offset += limit

    df = pd.DataFrame(tracks_data)
    return df if df.empty else df.drop_duplicates(subset=['track_id'])

def get_saved_tracks():
    """Extrai as Músicas Curtidas do usuário (A prova de 403)."""
    tracks_data = []
    limit = 50
    offset = 0

    print("Iniciando extração das Músicas Curtidas...")
    while True:
        results = sp.current_user_saved_tracks(limit=limit, offset=offset)
        tracks = results['items']

        if not tracks:
            break

        for item in tracks:
            track = item['track']
            if not track or track.get('is_local') or not track.get('id'):
                continue

            artist_id = track['artists'][0]['id'] if track.get('artists') else None
            tracks_data.append({
                'source': 'Saved Tracks',
                'track_id': track['id'],
                'track_name': track['name'],
                'artist_id': artist_id,
                'artist_name': track['artists'][0]['name'] if track.get('artists') else "Unknown"
            })

        offset += limit

    return pd.DataFrame(tracks_data)

# ==========================================
# 3. ENRIQUECIMENTO DOS DADOS
# ==========================================
def get_artist_details(df):
    """Busca gêneros e popularidade baseados no artista principal em lotes de 50."""
    print("\nBuscando metadados dos artistas (Gêneros e Popularidade)...")

    # Remove nulos e pega IDs únicos
    unique_artists = df['artist_id'].dropna().unique().tolist()
    artist_data_map = {}

    for i in range(0, len(unique_artists), 50):
        batch = unique_artists[i:i+50]
        try:
            artists_data = sp.artists(batch)
            for artist in artists_data['artists']:
                if artist:
                    artist_data_map[artist['id']] = {
                        'genres': ", ".join(artist['genres']) if artist['genres'] else "Unknown",
                        'artist_popularity': artist['popularity']
                    }
        except Exception as e:
            print(f"Erro ao buscar lote de artistas: {e}")
        time.sleep(0.5) # Respeitando o rate limit

    # Mapeando os novos dados de volta para o DataFrame original
    df['genres'] = df['artist_id'].map(lambda x: artist_data_map.get(x, {}).get('genres', 'Unknown'))
    df['artist_popularity'] = df['artist_id'].map(lambda x: artist_data_map.get(x, {}).get('artist_popularity', 0))

    return df

# ==========================================
# 4. EXECUÇÃO DO PIPELINE
# ==========================================
if __name__ == "__main__":
    print("Buscando playlists...")
    my_playlists = get_user_playlists()

    print("Extraindo músicas das playlists...")
    df_playlists = get_tracks_from_playlists(my_playlists)

    print("\nBuscando Músicas Curtidas...")
    df_saved = get_saved_tracks()

    frames = [df for df in [df_playlists, df_saved] if not df.empty]

    if not frames:
        print("\n❌ Nenhuma música foi extraída. O pipeline foi encerrado com segurança.")
        sys.exit()

    df_tracks = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['track_id'])
    print(f"\nTotal de músicas únicas consolidadas: {len(df_tracks)}")

    # Nova etapa de enriquecimento usando a API de Artistas
    df_final = get_artist_details(df_tracks)

    # Salvando os dados prontos para o Colab/Pandas
    df_final.to_csv("meu_dataset_spotify_atualizado.csv", index=False)
    print("\n✅ Extração concluída! Dados salvos em 'meu_dataset_spotify_atualizado.csv'")